# Cultivating T states in seconds

Reproduces *"Comparing Cultivation of T|+⟩, S|+⟩, and Z|+⟩ States"* from Gidney, Shutty and Jones,
[*Magic state cultivation*](https://arxiv.org/abs/2409.17595) (2024): distance-3 cultivation of a T state, against the same
circuit cultivating an S or a Z state, under uniform depolarizing noise. Every detector is postselected, so there is no
decoder: a kept shot is a logical error when its observable flips.

The T circuit is not Clifford. The paper sampled it with a state-vector simulator, and its 18 points took 7.2 CPU-years.
Tsim samples them with the same API as Stim; on an Apple GPU, it takes seconds. `cultivation.py` holds the circuits'
paths, the shot budget and the plot.

In [ ]:
import time

import numpy as np
import tsim

from cultivation import BATCH, batches, circuit_path, plot, points

start = time.perf_counter()
tally = points()  # T, S and Z at six noise strengths
for point in tally:
    circuit = tsim.Circuit.from_file(circuit_path(point))
    sampler = circuit.compile_detector_sampler()
    mask = np.ones(circuit.num_detectors, dtype=bool)  # keep shots where no detector fires
    for _ in batches(point):
        detectors, observables = sampler.sample(
            BATCH, postselection_mask=mask, separate_observables=True, bit_packed=True
        )
        point.shots += BATCH
        point.kept += len(observables)
        point.errors += np.count_nonzero(observables)  # the observable flipped
seconds = time.perf_counter() - start
print(sampler)
print(f"{sum(p.shots for p in tally) / 1e9:.1f} billion shots in {seconds:.1f} s")

In [ ]:
plot(tally, seconds)